# 11 - Expanded-data arrival-delay models and prediction

This notebook continues the earlier model flow using notebook 10 Parquet.
It compares the historical baseline, Ridge, Random Forest, Gradient-Boosted
Trees, XGBoost and CatBoost under one temporal contract.

Model selection uses December 2022 only. March and June 2023 remain locked
until a winner is frozen. The regression task is measured with MAE, RMSE,
median absolute error and p90 absolute error. A parallel classification task
compares whether arrival delay exceeds 15 minutes or 1 minute and reports accuracy,
balanced accuracy, precision, recall, F1, ROC-AUC, PR-AUC and confusion counts.

In [1]:
from pathlib import Path
import gc
import json
import sys
import warnings
import joblib
import numpy as np
import pandas as pd
from sklearn.ensemble import HistGradientBoostingRegressor, RandomForestRegressor
from sklearn.feature_extraction import FeatureHasher
from sklearn.exceptions import ConvergenceWarning
from sklearn.linear_model import LogisticRegression, Ridge, SGDClassifier

PROJECT_ROOT = Path.cwd().resolve().parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))
from src.t60_modeling import (
    HistoricalMedianBaseline, MixedCategoricalRidgePreprocessor,
    add_schedule_features, align_predictions_to_index,
    classification_threshold_table, compact_score, delay_classification_metrics,
    haul_direction_classification_metrics, haul_direction_segment_metrics,
    hurdle_expected_delay, segment_metrics, weighted_prediction_blend,
)

DATA_ROOT = PROJECT_ROOT / "data" / "processed" / "expanded_arrival_pre_t60"
REPORT_ROOT = PROJECT_ROOT / "reports" / "modeling" / "expanded"
MODEL_ROOT = PROJECT_ROOT / "models" / "expanded"
REPORT_ROOT.mkdir(parents=True, exist_ok=True)
MODEL_ROOT.mkdir(parents=True, exist_ok=True)
TARGET = "Arrival_Delay_Min"
DELAY_THRESHOLD_MINUTES = 15.0
ALTERNATIVE_DELAY_THRESHOLD_MINUTES = 1.0
CLASSIFICATION_TARGETS = (DELAY_THRESHOLD_MINUTES, ALTERNATIVE_DELAY_THRESHOLD_MINUTES)
THRESHOLD_SELECTION_OBJECTIVE = "f1"  # alternatives: balanced_accuracy, cost_per_flight
FALSE_NEGATIVE_COST = 3.0
FALSE_POSITIVE_COST = 1.0
LOGISTIC_TOLERANCE = 1e-3
LOGISTIC_INITIAL_MAX_ITER = 300  # retained only for the disabled legacy block
LOGISTIC_MAX_ITER = 300
FINAL_LOGISTIC_MAX_ITER = 100
CLASSIFIER_TUNING_PERCENT_OF_LOADED_TRAIN = 10
QUANTILE_LEVEL = 0.90
RUN_QUANTILE_MODEL = False  # severe >60-minute modelling is not a current priority
RUN_MODELS = True
RUN_REGRESSION_BENCHMARKS = False  # completed CSV benchmarks are reused
RUN_CLASSIFICATION = True
RUN_CATBOOST_CLASSIFIER = True
TRAIN_SAMPLE_PERCENT = 10
VALIDATION_SAMPLE_PERCENT = 5
TREE_SAMPLE_PERCENT = 1
SEED = 42

## 1. Load frozen splits and audit leakage

Notebook 10 must first write Parquet. Sample percentages control resources,
not dates. Test labels are loaded for final scoring only after validation
freezes the winner.

In [2]:
def deterministic_sample(frame, percent):
    if percent >= 100:
        return frame.copy()
    bucket = pd.util.hash_pandas_object(frame["ECTRL ID"], index=False) % 100
    return frame.loc[bucket < percent].copy()

if RUN_MODELS:
    development_splits = {
        name: add_schedule_features(pd.read_parquet(DATA_ROOT / name))
        for name in ("train", "validation")
    }
    forbidden = {"ACTUAL OFF BLOCK TIME", "ACTUAL ARRIVAL TIME",
                 "Departure_Delay_Min", "Actual Distance Flown (nm)"}
    assert all(not (forbidden & set(frame.columns)) for frame in development_splits.values())
    cutoff = (pd.to_datetime(development_splits["train"]["FILED OFF BLOCK TIME"])
              - pd.to_datetime(development_splits["train"]["prediction_cutoff_t60"])
             ).dt.total_seconds()/60
    assert np.allclose(cutoff, 60)
    train = deterministic_sample(development_splits["train"], TRAIN_SAMPLE_PERCENT)
    validation = deterministic_sample(
        development_splits["validation"], VALIDATION_SAMPLE_PERCENT
    )
    print({
        **{name: len(frame) for name, frame in development_splits.items()},
        "locked_test_partitions_read": False,
    })
else:
    print("Set RUN_MODELS=True after notebook 10 writes Parquet.")

{'train': 3373965, 'validation': 591391, 'locked_test_partitions_read': False}


## 2. Shared features and historical baseline

Airports, operator and grouped aircraft are hashed for Ridge and native
categories for CatBoost. Numeric medians and scaling are learned on train.
Operational T-60 columns are included automatically if notebook 10 built them.
The model also receives origin/destination continent, transatlantic direction,
short/medium versus long-haul band, scheduled arrival hour and duration-by-direction
interactions. Separate regression and classification tables are exported for <=3h,
3-6h, <=6h combined, >6h,
Europe-to-Americas and Americas-to-Europe flights.

The baseline fallback is route+airline, route, departure-airport+airline,
departure airport and global median, all fitted on train.

In [3]:
LOW_CARDINALITY_COLUMNS = [
    "STATFOR Market Segment", "Class_aircraft", "Number+Engine Type_aircraft",
    "ADEP_Continent", "ADES_Continent", "Duration_Band",
    "Transatlantic_Direction",
]
HIGH_CARDINALITY_COLUMNS = [
    "ADEP", "ADES", "AC Operator", "AC Type_grouped",
]
CATEGORICAL_COLUMNS = LOW_CARDINALITY_COLUMNS + HIGH_CARDINALITY_COLUMNS
STATIC_NUMERIC_COLUMNS = [
    "Requested_FL_Imputed", "scheduled_duration_min",
    "departure_hour_sin", "departure_hour_cos",
    "departure_dow_sin", "departure_dow_cos", "departure_month",
    "scheduled_arrival_hour_sin", "scheduled_arrival_hour_cos",
    "Is_Transatlantic", "duration_x_europe_to_americas",
    "duration_x_americas_to_europe",
]
if RUN_MODELS:
    OPERATIONAL_COLUMNS = [
        column for column in train.columns
        if column.startswith(("adep_dep_", "ades_arr_", "route_arr_",
                              "operator_dep_", "operator_arr_", "rotation_",
                              "adep_scheduled_", "ades_scheduled_",
                              "scheduled_airport_pressure_"))
    ]
    NUMERIC_COLUMNS = STATIC_NUMERIC_COLUMNS + OPERATIONAL_COLUMNS
    baseline = HistoricalMedianBaseline().fit(train)
    validation_predictions = {
        "historical_baseline": baseline.predict(validation)
    }
    SELECTED_HYPERPARAMETERS = {}
    print({"numeric_features": len(NUMERIC_COLUMNS),
           "operational_features": len(OPERATIONAL_COLUMNS)})

{'numeric_features': 80, 'operational_features': 68}


## 3. Ridge selection

Ridge remains the primary balanced model: it is memory-efficient, stable with
high-cardinality hashing and was comparatively strong for delayed flights.
Alpha is selected on validation only. The final Ridge contract excludes raw
aircraft registration because its previous improvement was negligible. Rotation
history and accumulated-delay variables remain, preserving operational continuity
without memorising a high-cardinality aircraft identifier.

In [4]:
if RUN_MODELS:
    # The final linear model intentionally excludes the aircraft registration.
    ridge_variant_columns = {
        "ridge_without_registration": HIGH_CARDINALITY_COLUMNS,
    }
    ridge_rows, ridge_models, ridge_preprocessors = [], {}, {}
    for variant, high_cardinality_columns in ridge_variant_columns.items():
        preprocessor = MixedCategoricalRidgePreprocessor(
            LOW_CARDINALITY_COLUMNS, high_cardinality_columns, NUMERIC_COLUMNS
        )
        x_ridge_train = preprocessor.fit_transform(train)
        x_ridge_validation = preprocessor.transform(validation)
        variant_results = []
        for alpha in (0.1, 1.0, 10.0, 100.0):
            model = Ridge(alpha=alpha, solver="lsqr").fit(
                x_ridge_train, train[TARGET]
            )
            prediction = model.predict(x_ridge_validation)
            metrics = segment_metrics(
                validation[TARGET], prediction, variant, "validation"
            )
            score = compact_score(metrics)
            ridge_rows.append({"variant": variant, "alpha": alpha, **score})
            variant_results.append((score["combined_MAE_score"],
                                    score["global_MAE"], alpha, model, prediction))
        best = min(variant_results, key=lambda item: (item[0], item[1]))
        ridge_models[variant] = best[3]
        ridge_preprocessors[variant] = preprocessor
        validation_predictions[variant] = best[4]
        SELECTED_HYPERPARAMETERS[variant] = {
            "alpha": float(best[2]),
            "includes_raw_registration": False,
        }
        del x_ridge_train, x_ridge_validation
        gc.collect()
    ridge_selection = pd.DataFrame(ridge_rows).sort_values(
        ["variant", "combined_MAE_score", "global_MAE"]
    )
    ridge_selection.to_csv(
        REPORT_ROOT / "ridge_hyperparameter_selection.csv", index=False
    )
    display(ridge_selection)

,variant,alpha,global_MAE,delayed_MAE,combined_MAE_score
3,ridge,100.0,9.749329,18.172981,13.961155
2,ridge,10.0,9.808994,18.124679,13.966837
1,ridge,1.0,9.883809,18.198163,14.040986
0,ridge,0.1,9.902804,18.217581,14.060192
5,ridge_without_registration,1.0,9.761590,18.178242,13.969916
6,ridge_without_registration,10.0,9.753009,18.188279,13.970644
4,ridge_without_registration,0.1,9.763268,18.178570,13.970919
7,ridge_without_registration,100.0,9.753697,18.211663,13.982680


## 4. Random Forest, GBT and XGBoost

Tree models use a smaller deterministic train sample by default because they
need more RAM, but use identical validation rows. Three focused configurations
per family vary depth/leaf regularisation and learning rate rather than running
an expensive exhaustive grid. XGBoost is the agreed fourth model.
LightGBM/SynapseML remains a future option with extra runtime complexity. The old
p90 severe-delay experiment is retained behind RUN_QUANTILE_MODEL=False, but is not
part of the current workflow because isolated >60-minute causes are often unobserved.

In [5]:
def dense_tree_frame(frame, medians=None):
    categorical = frame[CATEGORICAL_COLUMNS].fillna("__MISSING__").astype(str)
    hashed = FeatureHasher(
        n_features=512, input_type="string", alternate_sign=False
    ).transform(
        ([f"{column}={value}" for column, value in zip(CATEGORICAL_COLUMNS, row)]
         for row in categorical.itertuples(index=False, name=None))
    ).toarray()
    numeric = frame[NUMERIC_COLUMNS]
    medians = numeric.median() if medians is None else medians
    numeric = numeric.fillna(medians).to_numpy(dtype=np.float32)
    return np.hstack([hashed.astype(np.float32), numeric]), medians

def predict_dense_in_batches(model, frame, medians, batch_size=50000):
    predictions = []
    for start in range(0, len(frame), batch_size):
        matrix, _ = dense_tree_frame(frame.iloc[start:start + batch_size], medians)
        predictions.append(model.predict(matrix))
    return np.concatenate(predictions)

def quantile_risk_metrics(y_true, upper_prediction, scope):
    y = np.asarray(y_true, dtype=float)
    prediction = np.asarray(upper_prediction, dtype=float)
    residual = y - prediction
    pinball = np.maximum(QUANTILE_LEVEL * residual, (QUANTILE_LEVEL - 1) * residual)
    severe = y > 60.0
    flagged = prediction > 60.0
    return pd.DataFrame([{
        "evaluation_scope": scope, "quantile": QUANTILE_LEVEL,
        "rows": len(y), "pinball_loss": float(pinball.mean()),
        "empirical_coverage": float((y <= prediction).mean()),
        "actual_severe_rate": float(severe.mean()),
        "predicted_severe_risk_rate": float(flagged.mean()),
        "severe_recall": float((flagged & severe).sum() / severe.sum()) if severe.any() else np.nan,
        "severe_precision": float((flagged & severe).sum() / flagged.sum()) if flagged.any() else np.nan,
    }])

if RUN_MODELS and RUN_REGRESSION_BENCHMARKS:
    tree_train = deterministic_sample(train, TREE_SAMPLE_PERCENT)
    x_tree_train, tree_medians = dense_tree_frame(tree_train)
    x_tree_validation, _ = dense_tree_frame(validation, tree_medians)
    tree_candidate_specs = {
        "random_forest": [
            {"n_estimators": 150, "max_depth": 12, "min_samples_leaf": 5, "max_features": 0.7},
            {"n_estimators": 250, "max_depth": 18, "min_samples_leaf": 3, "max_features": 0.8},
            {"n_estimators": 200, "max_depth": None, "min_samples_leaf": 10, "max_features": 0.7},
        ],
        "gradient_boosted_trees": [
            {"max_iter": 250, "max_leaf_nodes": 31, "learning_rate": 0.05, "l2_regularization": 5, "min_samples_leaf": 20},
            {"max_iter": 350, "max_leaf_nodes": 31, "learning_rate": 0.03, "l2_regularization": 10, "min_samples_leaf": 20},
            {"max_iter": 250, "max_leaf_nodes": 63, "learning_rate": 0.04, "l2_regularization": 10, "min_samples_leaf": 30},
        ],
    }
    try:
        from xgboost import XGBRegressor
        tree_candidate_specs["xgboost"] = [
            {"n_estimators": 500, "max_depth": 6, "learning_rate": 0.04, "subsample": 0.8, "colsample_bytree": 0.8, "reg_lambda": 8},
            {"n_estimators": 700, "max_depth": 5, "learning_rate": 0.03, "subsample": 0.9, "colsample_bytree": 0.8, "reg_lambda": 12},
            {"n_estimators": 450, "max_depth": 7, "learning_rate": 0.035, "subsample": 0.8, "colsample_bytree": 0.7, "reg_lambda": 15},
        ]
    except ImportError:
        print("XGBoost unavailable; install project requirements.")

    tree_models, tree_tuning_rows = {}, []
    for family, candidates in tree_candidate_specs.items():
        family_results = []
        for candidate_id, parameters in enumerate(candidates, start=1):
            if family == "random_forest":
                model = RandomForestRegressor(
                    **parameters, n_jobs=2, random_state=SEED
                )
            elif family == "gradient_boosted_trees":
                model = HistGradientBoostingRegressor(
                    **parameters, random_state=SEED
                )
            else:
                model = XGBRegressor(
                    **parameters, objective="reg:absoluteerror",
                    n_jobs=2, random_state=SEED,
                )
            model.fit(x_tree_train, tree_train[TARGET])
            prediction = model.predict(x_tree_validation)
            score = compact_score(segment_metrics(
                validation[TARGET], prediction, family, "validation"
            ))
            row = {"family": family, "candidate": candidate_id,
                   "parameters": json.dumps(parameters), **score}
            tree_tuning_rows.append(row)
            family_results.append((score["combined_MAE_score"],
                                   score["global_MAE"], model, prediction, parameters))
        best = min(family_results, key=lambda item: (item[0], item[1]))
        tree_models[family] = best[2]
        validation_predictions[family] = best[3]
        SELECTED_HYPERPARAMETERS[family] = best[4]
    tree_hyperparameter_selection = pd.DataFrame(tree_tuning_rows).sort_values(
        ["family", "combined_MAE_score", "global_MAE"]
    )
    tree_hyperparameter_selection.to_csv(
        REPORT_ROOT / "tree_hyperparameter_selection.csv", index=False
    )
    display(tree_hyperparameter_selection)

    # Specialised >60-minute modelling is optional and disabled by default.
    # Those rows remain included in global and OTP15 evaluation.
    quantile_model = None
    if RUN_QUANTILE_MODEL:
        quantile_parameters = {
            "loss": "quantile", "quantile": QUANTILE_LEVEL,
            "max_iter": 350, "max_leaf_nodes": 31, "learning_rate": 0.05,
            "l2_regularization": 10, "min_samples_leaf": 30,
            "random_state": SEED,
        }
        quantile_model = HistGradientBoostingRegressor(**quantile_parameters).fit(
            x_tree_train, tree_train[TARGET]
        )
        quantile_validation_prediction = quantile_model.predict(x_tree_validation)
        quantile_validation_metrics = quantile_risk_metrics(
            validation[TARGET], quantile_validation_prediction, "validation"
        )
        quantile_validation_metrics.to_csv(
            REPORT_ROOT / "quantile_validation_metrics.csv", index=False
        )
        SELECTED_HYPERPARAMETERS["quantile_p90"] = quantile_parameters
        display(quantile_validation_metrics)
    else:
        print("Specialised >60-minute quantile model skipped.")
else:
    tree_models, tree_medians, quantile_model = {}, None, None
    print("Regression tree benchmarks reused from their completed CSVs.")

,family,candidate,parameters,global_MAE,delayed_MAE,combined_MAE_score
5,gradient_boosted_trees,3,"{""max_iter"": 250, ""max_leaf_nodes"": 63, ""learn...",9.960373,18.257013,14.108693
3,gradient_boosted_trees,1,"{""max_iter"": 250, ""max_leaf_nodes"": 31, ""learn...",9.994292,18.386086,14.190189
4,gradient_boosted_trees,2,"{""max_iter"": 350, ""max_leaf_nodes"": 31, ""learn...",9.971780,18.414717,14.193249
2,random_forest,3,"{""n_estimators"": 200, ""max_depth"": null, ""min_...",9.972829,18.195885,14.084357
1,random_forest,2,"{""n_estimators"": 250, ""max_depth"": 18, ""min_sa...",10.017760,18.261377,14.139569
0,random_forest,1,"{""n_estimators"": 150, ""max_depth"": 12, ""min_sa...",10.020490,18.459493,14.239992
6,xgboost,1,"{""n_estimators"": 500, ""max_depth"": 6, ""learnin...",9.706500,19.604762,14.655631
8,xgboost,3,"{""n_estimators"": 450, ""max_depth"": 7, ""learnin...",9.715551,19.668942,14.692247
7,xgboost,2,"{""n_estimators"": 700, ""max_depth"": 5, ""learnin...",9.732679,19.721889,14.727284


## 5. CatBoost and Ridge–CatBoost ensemble

CatBoost is the nonlinear categorical challenger. Native categories capture
route/operator/aircraft interactions without huge one-hot matrices.
Two focused depth/learning-rate configurations are compared on December.
Early stopping and time-aware fitting limit overfitting. Because CatBoost sorts
the validation frame for time-aware learning, predictions are explicitly restored
to the original row index before every metric is calculated. A weight grid on
temporal validation then blends Ridge and CatBoost using the agreed combined MAE.

**Corrected targeted rerun (21 August 2026).** The best CatBoost regressor
now records global MAE 9.431 and combined MAE 14.379; its implied delayed-flight
MAE is 19.327. The corrected classifier reaches PR-AUC 0.533. Selecting its
threshold on validation by F1 chooses 0.26, with F1 0.513, recall 0.616 and
precision 0.439. These corrected CSVs supersede the older embedded cell output.

In [6]:
if RUN_MODELS and RUN_REGRESSION_BENCHMARKS:
    try:
        from catboost import CatBoostRegressor, Pool
        cat_train = train.sort_values("FILED OFF BLOCK TIME").copy()
        cat_validation = validation.sort_values("FILED OFF BLOCK TIME").copy()
        cat_medians = cat_train[NUMERIC_COLUMNS].median()
        for frame in (cat_train, cat_validation):
            frame[CATEGORICAL_COLUMNS] = (
                frame[CATEGORICAL_COLUMNS].fillna("__MISSING__").astype(str)
            )
            frame[NUMERIC_COLUMNS] = frame[NUMERIC_COLUMNS].fillna(cat_medians)
        cat_features = CATEGORICAL_COLUMNS + NUMERIC_COLUMNS
        train_pool = Pool(cat_train[cat_features], cat_train[TARGET],
                          cat_features=CATEGORICAL_COLUMNS)
        validation_pool = Pool(cat_validation[cat_features], cat_validation[TARGET],
                               cat_features=CATEGORICAL_COLUMNS)
        catboost_candidates = [
            {"iterations": 900, "learning_rate": 0.05, "depth": 6, "l2_leaf_reg": 8},
            {"iterations": 1200, "learning_rate": 0.03, "depth": 7, "l2_leaf_reg": 10},
        ]
        catboost_results = []
        for candidate_id, parameters in enumerate(catboost_candidates, start=1):
            candidate_model = CatBoostRegressor(
                **parameters, loss_function="MAE", eval_metric="MAE",
                has_time=True, one_hot_max_size=10, max_ctr_complexity=2,
                random_seed=SEED, thread_count=2, od_type="Iter", od_wait=80,
                allow_writing_files=False, verbose=False,
            )
            candidate_model.fit(
                train_pool, eval_set=validation_pool, use_best_model=True
            )
            prediction_sorted = candidate_model.predict(validation_pool)
            prediction = align_predictions_to_index(
                prediction_sorted, cat_validation.index, validation.index
            )
            score = compact_score(segment_metrics(
                validation[TARGET], prediction, "catboost", "validation"
            ))
            catboost_results.append((score["combined_MAE_score"],
                                     score["global_MAE"], candidate_model,
                                     prediction, parameters, candidate_model.get_best_iteration()))
        best_catboost = min(catboost_results, key=lambda item: (item[0], item[1]))
        catboost_model = best_catboost[2]
        validation_predictions["catboost"] = best_catboost[3]
        SELECTED_HYPERPARAMETERS["catboost"] = {
            **best_catboost[4], "best_iteration": int(best_catboost[5])
        }
        catboost_hyperparameter_selection = pd.DataFrame([
            {"candidate": index + 1, "parameters": json.dumps(item[4]),
             "best_iteration": item[5], "combined_MAE_score": item[0],
             "global_MAE": item[1]}
            for index, item in enumerate(catboost_results)
        ]).sort_values(["combined_MAE_score", "global_MAE"])
        catboost_hyperparameter_selection.to_csv(
            REPORT_ROOT / "catboost_hyperparameter_selection.csv", index=False
        )
        display(catboost_hyperparameter_selection)
    except ImportError:
        print("CatBoost unavailable; install project requirements.")

    # Learn the Ridge/CatBoost blend only on temporal validation.
    if "catboost" in validation_predictions:
        ensemble_rows = []
        for ridge_weight in np.linspace(0.0, 1.0, 21):
            prediction = weighted_prediction_blend(
                validation_predictions["ridge_without_registration"],
                validation_predictions["catboost"],
                ridge_weight,
            )
            score = compact_score(segment_metrics(
                validation[TARGET], prediction, "ridge_catboost_ensemble",
                "validation",
            ))
            ensemble_rows.append({"ridge_weight": ridge_weight,
                                  "catboost_weight": 1.0 - ridge_weight,
                                  **score})
        ensemble_selection = pd.DataFrame(ensemble_rows).sort_values(
            ["combined_MAE_score", "global_MAE"]
        )
        best_ensemble = ensemble_selection.iloc[0]
        ENSEMBLE_RIDGE_WEIGHT = float(best_ensemble["ridge_weight"])
        validation_predictions["ridge_catboost_ensemble"] = (
            weighted_prediction_blend(
                validation_predictions["ridge_without_registration"],
                validation_predictions["catboost"],
                ENSEMBLE_RIDGE_WEIGHT,
            )
        )
        SELECTED_HYPERPARAMETERS["ridge_catboost_ensemble"] = {
            "ridge_weight": ENSEMBLE_RIDGE_WEIGHT,
            "catboost_weight": 1.0 - ENSEMBLE_RIDGE_WEIGHT,
            "selection_split": "validation_2022_12",
        }
        ensemble_selection.to_csv(
            REPORT_ROOT / "ridge_catboost_ensemble_weight_selection.csv",
            index=False,
        )
        display(ensemble_selection.head())
    else:
        print("Ensemble skipped because CatBoost regression is unavailable.")
else:
    print("CatBoost regression and ensemble skipped; completed CSVs are retained.")

,candidate,parameters,best_iteration,combined_MAE_score,global_MAE
1,2,"{""iterations"": 1200, ""learning_rate"": 0.03, ""d...",1199,20.064507,13.557233
0,1,"{""iterations"": 900, ""learning_rate"": 0.05, ""de...",899,20.064680,13.562133


## 6. Freeze winner on validation

Ranking gives equal weight to global MAE and MAE among flights delayed over 15
minutes. RMSE, median and p90 errors remain in the detailed segment report.
No test label is accessed here. The published deployment choice is explicitly
fixed to Ridge without aircraft registration; the other models remain benchmarks.

In [7]:
if RUN_MODELS:
    validation_metrics = pd.concat([
        segment_metrics(validation[TARGET], prediction, name, "validation")
        for name, prediction in validation_predictions.items()
    ], ignore_index=True)
    if not RUN_REGRESSION_BENCHMARKS:
        previous_metrics_path = REPORT_ROOT / "validation_segment_metrics.csv"
        if previous_metrics_path.exists():
            previous_metrics = pd.read_csv(previous_metrics_path)
            previous_metrics = previous_metrics[
                ~previous_metrics["model"].isin(
                    list(validation_predictions) + ["two_stage_hurdle"]
                )
                & previous_metrics["segment"].isin(
                    ["all", "punctual_<=15", "delayed_>15"]
                )
            ]
            validation_metrics = pd.concat(
                [validation_metrics, previous_metrics], ignore_index=True
            )
    validation_summary = pd.DataFrame([
        {"candidate": name, **compact_score(group)}
        for name, group in validation_metrics.groupby("model")
    ]).sort_values(["combined_MAE_score", "global_MAE"])
    validation_metrics.to_csv(
        REPORT_ROOT / "validation_segment_metrics.csv", index=False
    )
    validation_haul_direction_metrics = pd.concat([
        haul_direction_segment_metrics(
            validation, validation[TARGET], prediction, name, "validation"
        )
        for name, prediction in validation_predictions.items()
    ], ignore_index=True)
    if not RUN_REGRESSION_BENCHMARKS:
        previous_haul_path = REPORT_ROOT / "validation_haul_direction_metrics.csv"
        if previous_haul_path.exists():
            previous_haul = pd.read_csv(previous_haul_path)
            previous_haul = previous_haul[
                ~previous_haul["model"].isin(
                    list(validation_predictions) + ["two_stage_hurdle"]
                )
            ]
            validation_haul_direction_metrics = pd.concat(
                [validation_haul_direction_metrics, previous_haul],
                ignore_index=True,
            )
    validation_haul_direction_metrics.to_csv(
        REPORT_ROOT / "validation_haul_direction_metrics.csv", index=False
    )
    validation_summary.to_csv(
        REPORT_ROOT / "validation_model_ranking.csv", index=False
    )
    validation_summary[
        validation_summary["candidate"] == "ridge_without_registration"
    ].to_csv(
        REPORT_ROOT / "ridge_without_registration_selection.csv", index=False
    )
    # Deployment choice requested for the simpler, privacy-safer contract.
    SELECTED_MODEL = "ridge_without_registration"
    assert SELECTED_MODEL in set(validation_summary["candidate"])
    (REPORT_ROOT / "selection.json").write_text(
        json.dumps({
            "selected_model": SELECTED_MODEL,
            "selection_rule": "forced Ridge without AC Registration",
            "selected_hyperparameters": SELECTED_HYPERPARAMETERS.get(
                SELECTED_MODEL, {}
            ),
        }, indent=2),
        encoding="utf-8",
    )
    display(validation_summary)

,candidate,global_MAE,delayed_MAE,combined_MAE_score
4,ridge,9.749329,18.172981,13.961155
5,ridge_without_registration,9.761590,18.178242,13.969916
3,random_forest,9.972829,18.195885,14.084357
1,gradient_boosted_trees,9.960373,18.257013,14.108693
6,xgboost,9.706500,19.604762,14.655631
2,historical_baseline,10.379190,23.362434,16.870812
0,catboost,13.557233,26.571781,20.064507


## 7. Parallel delay classification and two-stage regression

Regression remains the primary minute estimate. This additional task answers a
second operational question: will arrival delay exceed 15 minutes? A parallel
experiment asks whether it exceeds 1 minute. Each label is
derived only from the target and never enters the feature matrix. Accuracy is
reported, but each probability threshold is selected on validation by F1 by
default; balanced accuracy or an explicit false-negative cost can be configured.
Balanced accuracy, precision, recall, F1, ROC-AUC, PR-AUC
and the complete confusion matrix prevent a majority-class prediction from looking
artificially strong. Scalable SGD logistic regression tunes two regularisation
levels and three class weights on a small stratified training subset. Every
candidate is checkpointed immediately; only the selected configuration is then
refitted on the complete loaded training sample. CatBoost reuses only its previously
winning configuration for the primary 15-minute task. The selected
OTP15 probability is combined with a Ridge severity model trained on operational
delays from 15 to 60 minutes. Its output is capped at 60 and remains separate from
the uncapped minute Ridge used for the complete population.


In [8]:
# Legacy fixed-threshold block retained in the executed history but disabled.
if False and RUN_MODELS and RUN_CLASSIFICATION:
    train_delay_label = (train[TARGET] > DELAY_THRESHOLD_MINUTES).astype(int)
    validation_delay_label = (
        validation[TARGET] > DELAY_THRESHOLD_MINUTES
    ).astype(int)
    train_delay_rate = float(train_delay_label.mean())
    classification_probabilities = {
        "majority_baseline": np.full(len(validation), train_delay_rate)
    }

    for large_matrix in ("x_train", "x_validation", "x_tree_train",
                         "x_tree_validation"):
        globals().pop(large_matrix, None)
    gc.collect()
    classifier_preprocessor = MixedCategoricalRidgePreprocessor(
        LOW_CARDINALITY_COLUMNS, HIGH_CARDINALITY_COLUMNS, NUMERIC_COLUMNS
    )
    x_classifier_train = classifier_preprocessor.fit_transform(train)
    x_classifier_validation = classifier_preprocessor.transform(validation)
    logistic_candidates = {}
    for c_value in (0.05, 0.2, 1.0):
        classifier = LogisticRegression(
            C=c_value, solver="saga", max_iter=300, random_state=SEED,
        )
        classifier.fit(x_classifier_train, train_delay_label)
        probability = classifier.predict_proba(x_classifier_validation)[:, 1]
        name = f"logistic_c{c_value:g}"
        logistic_candidates[name] = classifier
        classification_probabilities[name] = probability
    del x_classifier_train, x_classifier_validation
    gc.collect()

    try:
        if not RUN_CATBOOST_CLASSIFIER:
            raise ImportError("CatBoost classifier disabled by configuration")
        from catboost import CatBoostClassifier, Pool
        classification_train_pool = Pool(
            cat_train[cat_features], train_delay_label.loc[cat_train.index],
            cat_features=CATEGORICAL_COLUMNS,
        )
        classification_validation_pool = Pool(
            cat_validation[cat_features],
            validation_delay_label.loc[cat_validation.index],
            cat_features=CATEGORICAL_COLUMNS,
        )
        catboost_classifier_candidates = [
            {"iterations": 700, "learning_rate": 0.05, "depth": 6, "l2_leaf_reg": 8},
            {"iterations": 900, "learning_rate": 0.035, "depth": 7, "l2_leaf_reg": 10},
        ]
        catboost_classifier_results = []
        for candidate_id, parameters in enumerate(
            catboost_classifier_candidates, start=1
        ):
            candidate_classifier = CatBoostClassifier(
                **parameters, loss_function="Logloss", eval_metric="PRAUC",
                has_time=True, one_hot_max_size=10, max_ctr_complexity=2,
                random_seed=SEED, thread_count=2, od_type="Iter", od_wait=60,
                allow_writing_files=False, verbose=False,
            )
            candidate_classifier.fit(
                classification_train_pool,
                eval_set=classification_validation_pool, use_best_model=True,
            )
            probability = candidate_classifier.predict_proba(
                classification_validation_pool
            )[:, 1]
            metrics = delay_classification_metrics(
                validation[TARGET], probability, "catboost_classifier",
                "validation", delay_threshold_minutes=DELAY_THRESHOLD_MINUTES,
            ).iloc[0]
            catboost_classifier_results.append((
                metrics["average_precision"], metrics["f1"],
                metrics["balanced_accuracy"], candidate_classifier,
                probability, parameters, candidate_classifier.get_best_iteration(),
            ))
        best_catboost_classifier = max(
            catboost_classifier_results, key=lambda item: (item[0], item[1], item[2])
        )
        catboost_classifier = best_catboost_classifier[3]
        classification_probabilities["catboost_classifier"] = best_catboost_classifier[4]
        SELECTED_HYPERPARAMETERS["catboost_classifier"] = {
            **best_catboost_classifier[5],
            "best_iteration": int(best_catboost_classifier[6]),
        }
        pd.DataFrame([
            {"candidate": index + 1, "parameters": json.dumps(item[5]),
             "best_iteration": item[6], "average_precision": item[0],
             "f1": item[1], "balanced_accuracy": item[2]}
            for index, item in enumerate(catboost_classifier_results)
        ]).sort_values("average_precision", ascending=False).to_csv(
            REPORT_ROOT / "catboost_classifier_hyperparameter_selection.csv",
            index=False,
        )
    except (ImportError, NameError):
        print("CatBoost classification unavailable; logistic models remain valid.")

    classification_validation_metrics = pd.concat([
        delay_classification_metrics(
            validation[TARGET], probability, name, "validation",
            delay_threshold_minutes=DELAY_THRESHOLD_MINUTES,
        )
        for name, probability in classification_probabilities.items()
    ], ignore_index=True)
    regression_as_classification = delay_classification_metrics(
        validation[TARGET], validation_predictions[SELECTED_MODEL],
        f"{SELECTED_MODEL}_minutes_threshold", "validation",
        delay_threshold_minutes=DELAY_THRESHOLD_MINUTES,
        scores_are_probabilities=False,
    )
    classification_validation_metrics = pd.concat(
        [classification_validation_metrics, regression_as_classification],
        ignore_index=True,
    )
    eligible_classifiers = classification_validation_metrics[
        classification_validation_metrics["model"] !=
        f"{SELECTED_MODEL}_minutes_threshold"
    ]
    SELECTED_CLASSIFIER = eligible_classifiers.sort_values(
        ["average_precision", "f1", "balanced_accuracy"],
        ascending=False,
    ).iloc[0]["model"]
    if SELECTED_CLASSIFIER in logistic_candidates:
        SELECTED_HYPERPARAMETERS[SELECTED_CLASSIFIER] = {
            "C": float(SELECTED_CLASSIFIER.removeprefix("logistic_c")),
            "solver": "saga",
        }
    classification_validation_metrics.to_csv(
        REPORT_ROOT / "classification_validation_metrics.csv", index=False
    )
    classification_validation_metrics[
        classification_validation_metrics["model"].str.startswith("logistic_")
    ].sort_values("average_precision", ascending=False).to_csv(
        REPORT_ROOT / "logistic_hyperparameter_selection.csv", index=False
    )
    validation_classification_by_haul_direction = pd.concat([
        haul_direction_classification_metrics(
            validation, validation[TARGET], probability, name, "validation",
            delay_threshold_minutes=DELAY_THRESHOLD_MINUTES,
        )
        for name, probability in classification_probabilities.items()
    ], ignore_index=True)
    validation_classification_by_haul_direction.to_csv(
        REPORT_ROOT / "classification_validation_haul_direction_metrics.csv",
        index=False,
    )
    display(classification_validation_metrics.sort_values(
        "average_precision", ascending=False
    ))
    print({"selected_classifier": SELECTED_CLASSIFIER,
           "train_delay_rate": train_delay_rate})

def stratified_deterministic_sample(frame, target_threshold, percent):
    if percent >= 100:
        return frame.copy()
    labels = frame[TARGET] > target_threshold
    parts = [deterministic_sample(frame.loc[labels == value], percent)
             for value in (False, True)]
    return pd.concat(parts).sort_index()

def build_sgd_logistic(alpha, class_weight, max_iter):
    # SGD optimises the same logistic objective but scales to millions of rows.
    return SGDClassifier(
        loss="log_loss", penalty="l2", alpha=alpha,
        class_weight=class_weight, max_iter=max_iter,
        tol=LOGISTIC_TOLERANCE, early_stopping=True,
        validation_fraction=0.1, n_iter_no_change=5, average=True,
        random_state=SEED, n_jobs=2,
    )

def best_threshold_row(table):
    if THRESHOLD_SELECTION_OBJECTIVE == "cost_per_flight":
        return table.sort_values(
            ["cost_per_flight", "f1", "balanced_accuracy"],
            ascending=[True, False, False],
        ).iloc[0]
    if THRESHOLD_SELECTION_OBJECTIVE not in {"f1", "balanced_accuracy"}:
        raise ValueError("Unsupported THRESHOLD_SELECTION_OBJECTIVE")
    return table.sort_values(
        [THRESHOLD_SELECTION_OBJECTIVE, "cost_per_flight", "average_precision"],
        ascending=[False, True, False],
    ).iloc[0]

if RUN_MODELS and RUN_CLASSIFICATION:
    classifier_checkpoint_dir = MODEL_ROOT / "checkpoints" / "classification"
    classifier_checkpoint_dir.mkdir(parents=True, exist_ok=True)
    primary_tuning_train = stratified_deterministic_sample(
        train, DELAY_THRESHOLD_MINUTES, CLASSIFIER_TUNING_PERCENT_OF_LOADED_TRAIN
    )
    classifier_preprocessor = MixedCategoricalRidgePreprocessor(
        LOW_CARDINALITY_COLUMNS, HIGH_CARDINALITY_COLUMNS, NUMERIC_COLUMNS
    )
    x_classifier_seed = classifier_preprocessor.fit_transform(primary_tuning_train)
    del x_classifier_seed, primary_tuning_train
    x_classifier_validation = classifier_preprocessor.transform(validation)
    logistic_alpha_values = (1e-5, 1e-4)
    class_weight_options = {
        "none": None, "balanced": "balanced",
        "positive_2x": {0: 1.0, 1: 2.0},
    }
    CLASSIFICATION_MODELS_BY_TARGET = {}
    CLASSIFICATION_PROBABILITIES_BY_TARGET = {}
    TRAIN_DELAY_RATES_BY_TARGET = {}
    logistic_convergence_rows = []
    logistic_candidate_checkpoint_rows = []
    LOGISTIC_SPECS_BY_TARGET = {}

    for target_threshold in CLASSIFICATION_TARGETS:
        train_labels = (train[TARGET] > target_threshold).astype(int)
        target_code = f"gt{target_threshold:g}min"
        train_rate = float(train_labels.mean())
        TRAIN_DELAY_RATES_BY_TARGET[target_threshold] = train_rate
        models = {}
        probabilities = {
            f"majority_baseline_{target_code}": np.full(len(validation), train_rate)
        }
        tuning_train = stratified_deterministic_sample(
            train, target_threshold, CLASSIFIER_TUNING_PERCENT_OF_LOADED_TRAIN
        )
        tuning_labels = (tuning_train[TARGET] > target_threshold).astype(int)
        x_classifier_tuning = classifier_preprocessor.transform(tuning_train)
        specs = {}
        for alpha in logistic_alpha_values:
            for weight_name, class_weight in class_weight_options.items():
                name = f"logistic_sgd_{target_code}_a{alpha:g}_{weight_name}"
                checkpoint_path = (classifier_checkpoint_dir / f"{name}.joblib")
                if checkpoint_path.exists():
                    classifier = joblib.load(checkpoint_path)
                    checkpoint_status = "loaded"
                else:
                    classifier = build_sgd_logistic(
                        alpha, class_weight, LOGISTIC_MAX_ITER
                    )
                    classifier.fit(x_classifier_tuning, tuning_labels)
                    joblib.dump(classifier, checkpoint_path)
                    checkpoint_status = "trained"
                models[name] = classifier
                probabilities[name] = classifier.predict_proba(
                    x_classifier_validation
                )[:, 1]
                specs[name] = {"alpha": alpha, "class_weight": class_weight,
                               "class_weight_name": weight_name}
                iterations_used = int(classifier.n_iter_)
                logistic_convergence_rows.append({
                    "target_threshold_minutes": target_threshold,
                    "model": name, "alpha": alpha,
                    "class_weight": weight_name,
                    "tolerance": LOGISTIC_TOLERANCE,
                    "max_iter": LOGISTIC_MAX_ITER,
                    "iterations_used": iterations_used,
                    "converged": iterations_used < LOGISTIC_MAX_ITER,
                    "checkpoint_status": checkpoint_status,
                    "tuning_rows": len(tuning_train),
                })
                checkpoint_metric = delay_classification_metrics(
                    validation[TARGET], probabilities[name], name, "validation",
                    delay_threshold_minutes=target_threshold,
                ).iloc[0].to_dict()
                checkpoint_metric.update({"alpha": alpha,
                                          "class_weight": weight_name,
                                          "checkpoint_status": checkpoint_status})
                logistic_candidate_checkpoint_rows.append(checkpoint_metric)
                pd.DataFrame(logistic_candidate_checkpoint_rows).to_csv(
                    REPORT_ROOT / "classification_candidate_checkpoint.csv",
                    index=False,
                )
                SELECTED_HYPERPARAMETERS[name] = {
                    "alpha": alpha, "solver": "sgd_log_loss",
                    "class_weight": weight_name,
                    "tolerance": LOGISTIC_TOLERANCE,
                    "max_iter": LOGISTIC_MAX_ITER,
                    "iterations_used": iterations_used,
                    "tuning_percent_of_loaded_train": (
                        CLASSIFIER_TUNING_PERCENT_OF_LOADED_TRAIN
                    ),
                    "target_threshold_minutes": target_threshold,
                }
        CLASSIFICATION_MODELS_BY_TARGET[target_threshold] = models
        CLASSIFICATION_PROBABILITIES_BY_TARGET[target_threshold] = probabilities
        LOGISTIC_SPECS_BY_TARGET[target_threshold] = specs
        del x_classifier_tuning, tuning_train, tuning_labels
        gc.collect()

    gc.collect()

    # Refit only the previously winning CatBoost classification configuration.
    try:
        if not RUN_CATBOOST_CLASSIFIER:
            raise ImportError("CatBoost classifier disabled by configuration")
        from catboost import CatBoostClassifier, Pool
        # Build classifier-specific CatBoost frames even when regression
        # benchmarks are skipped and their in-memory objects do not exist.
        cat_train = train.sort_values("FILED OFF BLOCK TIME").copy()
        cat_validation = validation.sort_values("FILED OFF BLOCK TIME").copy()
        cat_medians = cat_train[NUMERIC_COLUMNS].median()
        for frame in (cat_train, cat_validation):
            frame[CATEGORICAL_COLUMNS] = (
                frame[CATEGORICAL_COLUMNS].fillna("__MISSING__").astype(str)
            )
            frame[NUMERIC_COLUMNS] = frame[NUMERIC_COLUMNS].fillna(cat_medians)
        cat_features = CATEGORICAL_COLUMNS + NUMERIC_COLUMNS
        primary_train_labels = (train[TARGET] > DELAY_THRESHOLD_MINUTES).astype(int)
        primary_validation_labels = (validation[TARGET] > DELAY_THRESHOLD_MINUTES).astype(int)
        classification_train_pool = Pool(
            cat_train[cat_features], primary_train_labels.loc[cat_train.index],
            cat_features=CATEGORICAL_COLUMNS,
        )
        classification_validation_pool = Pool(
            cat_validation[cat_features],
            primary_validation_labels.loc[cat_validation.index],
            cat_features=CATEGORICAL_COLUMNS,
        )
        catboost_classifier_candidates = [
            {"iterations": 900, "learning_rate": 0.035, "depth": 7, "l2_leaf_reg": 10},
        ]
        catboost_classifier_results = []
        for candidate_id, parameters in enumerate(catboost_classifier_candidates, start=1):
            candidate_classifier = CatBoostClassifier(
                **parameters, loss_function="Logloss", eval_metric="PRAUC",
                has_time=True, one_hot_max_size=10, max_ctr_complexity=2,
                random_seed=SEED, thread_count=2, od_type="Iter", od_wait=60,
                allow_writing_files=False, verbose=False,
            )
            candidate_classifier.fit(
                classification_train_pool, eval_set=classification_validation_pool,
                use_best_model=True,
            )
            probability_sorted = candidate_classifier.predict_proba(
                classification_validation_pool
            )[:, 1]
            probability = align_predictions_to_index(
                probability_sorted, cat_validation.index, validation.index
            )
            fixed_metrics = delay_classification_metrics(
                validation[TARGET], probability, "catboost_classifier",
                "validation", delay_threshold_minutes=DELAY_THRESHOLD_MINUTES,
            ).iloc[0]
            catboost_classifier_results.append((
                fixed_metrics["average_precision"], fixed_metrics["f1"],
                candidate_classifier, probability, parameters,
                candidate_classifier.get_best_iteration(),
            ))
        best_catboost_classifier = max(
            catboost_classifier_results, key=lambda item: (item[0], item[1])
        )
        catboost_classifier = best_catboost_classifier[2]
        CLASSIFICATION_MODELS_BY_TARGET[DELAY_THRESHOLD_MINUTES][
            "catboost_classifier"
        ] = catboost_classifier
        CLASSIFICATION_PROBABILITIES_BY_TARGET[DELAY_THRESHOLD_MINUTES][
            "catboost_classifier"
        ] = best_catboost_classifier[3]
        SELECTED_HYPERPARAMETERS["catboost_classifier"] = {
            **best_catboost_classifier[4],
            "best_iteration": int(best_catboost_classifier[5]),
            "alignment": "restored_to_validation_index",
        }
        pd.DataFrame([
            {"candidate": index + 1, "parameters": json.dumps(item[4]),
             "best_iteration": item[5], "average_precision": item[0],
             "f1_at_0_5": item[1]}
            for index, item in enumerate(catboost_classifier_results)
        ]).sort_values("average_precision", ascending=False).to_csv(
            REPORT_ROOT / "catboost_classifier_hyperparameter_selection.csv",
            index=False,
        )
    except (ImportError, NameError):
        print("CatBoost classification unavailable; logistic models remain valid.")

    threshold_tables = []
    best_candidate_rows = []
    selected_classifiers_by_target = {}
    validation_segment_rows = []
    for target_threshold in CLASSIFICATION_TARGETS:
        probabilities = CLASSIFICATION_PROBABILITIES_BY_TARGET[target_threshold]
        target_best_rows = []
        for name, probability in probabilities.items():
            if name.startswith("majority_baseline"):
                row = delay_classification_metrics(
                    validation[TARGET], probability, name, "validation",
                    delay_threshold_minutes=target_threshold,
                ).iloc[0].to_dict()
                row["cost_per_flight"] = (
                    FALSE_NEGATIVE_COST * row["false_negative"]
                    + FALSE_POSITIVE_COST * row["false_positive"]
                ) / row["rows"]
            else:
                table = classification_threshold_table(
                    validation[TARGET], probability, name, "validation",
                    delay_threshold_minutes=target_threshold,
                    false_negative_cost=FALSE_NEGATIVE_COST,
                    false_positive_cost=FALSE_POSITIVE_COST,
                )
                table.insert(1, "target_threshold_minutes", target_threshold)
                threshold_tables.append(table)
                row = best_threshold_row(table).to_dict()
            row["target_threshold_minutes"] = target_threshold
            target_best_rows.append(row)
            best_candidate_rows.append(row)

        eligible = pd.DataFrame(target_best_rows)
        eligible = eligible[~eligible["model"].str.startswith("majority_baseline")]
        selected = best_threshold_row(eligible).to_dict()
        selected_tuning_name = selected["model"]
        if selected_tuning_name in LOGISTIC_SPECS_BY_TARGET[target_threshold]:
            spec = LOGISTIC_SPECS_BY_TARGET[target_threshold][selected_tuning_name]
            final_name = f"{selected_tuning_name}_full_refit"
            final_checkpoint = classifier_checkpoint_dir / f"{final_name}.joblib"
            if final_checkpoint.exists():
                final_classifier = joblib.load(final_checkpoint)
                final_status = "loaded"
            else:
                (REPORT_ROOT / "classification_fit_status.json").write_text(
                    json.dumps({"status": "fitting_full_selected_model",
                                "target_threshold_minutes": target_threshold,
                                "model": final_name, "rows": len(train)}, indent=2),
                    encoding="utf-8",
                )
                x_classifier_full = classifier_preprocessor.transform(train)
                final_classifier = build_sgd_logistic(
                    spec["alpha"], spec["class_weight"],
                    FINAL_LOGISTIC_MAX_ITER,
                )
                final_classifier.fit(x_classifier_full, train_labels)
                joblib.dump(final_classifier, final_checkpoint)
                del x_classifier_full
                gc.collect()
                final_status = "trained"
            final_probability = final_classifier.predict_proba(
                x_classifier_validation
            )[:, 1]
            models[final_name] = final_classifier
            probabilities[final_name] = final_probability
            CLASSIFICATION_MODELS_BY_TARGET[target_threshold] = models
            CLASSIFICATION_PROBABILITIES_BY_TARGET[target_threshold] = probabilities
            SELECTED_HYPERPARAMETERS[final_name] = {
                **SELECTED_HYPERPARAMETERS[selected_tuning_name],
                "max_iter": FINAL_LOGISTIC_MAX_ITER,
                "iterations_used": int(final_classifier.n_iter_),
                "final_fit_rows": len(train),
                "checkpoint_status": final_status,
            }
            final_table = classification_threshold_table(
                validation[TARGET], final_probability, final_name, "validation",
                delay_threshold_minutes=target_threshold,
                false_negative_cost=FALSE_NEGATIVE_COST,
                false_positive_cost=FALSE_POSITIVE_COST,
            )
            final_table.insert(1, "target_threshold_minutes", target_threshold)
            threshold_tables.append(final_table)
            selected = best_threshold_row(final_table).to_dict()
            selected["target_threshold_minutes"] = target_threshold
            best_candidate_rows.append(selected)
            (REPORT_ROOT / "classification_fit_status.json").write_text(
                json.dumps({"status": "complete",
                            "target_threshold_minutes": target_threshold,
                            "model": final_name,
                            "iterations_used": int(final_classifier.n_iter_)}, indent=2),
                encoding="utf-8",
            )
        selected_classifiers_by_target[target_threshold] = {
            "model": selected["model"],
            "probability_threshold": float(selected["decision_threshold"]),
            "selection_objective": THRESHOLD_SELECTION_OBJECTIVE,
        }
        selected_probability = probabilities[selected["model"]]
        validation_segment_rows.append(haul_direction_classification_metrics(
            validation, validation[TARGET], selected_probability,
            selected["model"], "validation",
            delay_threshold_minutes=target_threshold,
            decision_threshold=float(selected["decision_threshold"]),
        ))

        minute_row = delay_classification_metrics(
            validation[TARGET], validation_predictions[SELECTED_MODEL],
            f"{SELECTED_MODEL}_minutes_gt{target_threshold:g}", "validation",
            delay_threshold_minutes=target_threshold, scores_are_probabilities=False,
        ).iloc[0].to_dict()
        minute_row["target_threshold_minutes"] = target_threshold
        minute_row["cost_per_flight"] = (
            FALSE_NEGATIVE_COST * minute_row["false_negative"]
            + FALSE_POSITIVE_COST * minute_row["false_positive"]
        ) / minute_row["rows"]
        best_candidate_rows.append(minute_row)

    classification_threshold_grid = pd.concat(threshold_tables, ignore_index=True)
    classification_threshold_grid.to_csv(
        REPORT_ROOT / "classification_threshold_grid.csv", index=False
    )
    classification_validation_metrics = pd.DataFrame(best_candidate_rows)
    classification_validation_metrics.to_csv(
        REPORT_ROOT / "classification_validation_metrics.csv", index=False
    )
    classification_validation_metrics.to_csv(
        REPORT_ROOT / "classification_target_comparison.csv", index=False
    )
    classification_validation_metrics[
        classification_validation_metrics["model"].str.startswith("logistic_")
    ].to_csv(
        REPORT_ROOT / "logistic_hyperparameter_selection.csv", index=False
    )
    pd.DataFrame(logistic_convergence_rows).to_csv(
        REPORT_ROOT / "logistic_convergence.csv", index=False
    )
    pd.concat(validation_segment_rows, ignore_index=True).to_csv(
        REPORT_ROOT / "classification_validation_haul_direction_metrics.csv",
        index=False,
    )
    SELECTED_CLASSIFIERS_BY_TARGET = selected_classifiers_by_target
    primary_selection = SELECTED_CLASSIFIERS_BY_TARGET[DELAY_THRESHOLD_MINUTES]
    SELECTED_CLASSIFIER = primary_selection["model"]
    SELECTED_CLASSIFIER_THRESHOLD = primary_selection["probability_threshold"]
    train_delay_rate = TRAIN_DELAY_RATES_BY_TARGET[DELAY_THRESHOLD_MINUTES]
    logistic_candidates = CLASSIFICATION_MODELS_BY_TARGET[DELAY_THRESHOLD_MINUTES]
    (REPORT_ROOT / "classification_selection.json").write_text(
        json.dumps({
            "threshold_objective": THRESHOLD_SELECTION_OBJECTIVE,
            "false_negative_cost": FALSE_NEGATIVE_COST,
            "false_positive_cost": FALSE_POSITIVE_COST,
            "selected_by_target": {
                f"gt{target:g}min": selected
                for target, selected in SELECTED_CLASSIFIERS_BY_TARGET.items()
            },
        }, indent=2),
        encoding="utf-8",
    )
    display(classification_validation_metrics.sort_values(
        ["target_threshold_minutes", THRESHOLD_SELECTION_OBJECTIVE],
        ascending=[False, False],
    ))
    print({"selected_classifiers": SELECTED_CLASSIFIERS_BY_TARGET})
    del x_classifier_validation
    for temporary_name in ("classification_train_pool",
                           "classification_validation_pool",
                           "cat_train", "cat_validation"):
        globals().pop(temporary_name, None)
    gc.collect()

    # Two-stage hurdle model: P(delay > 15) x conditional delayed severity.
    primary_probability = CLASSIFICATION_PROBABILITIES_BY_TARGET[
        DELAY_THRESHOLD_MINUTES
    ][SELECTED_CLASSIFIER]
    delayed_train = train[(train[TARGET] > DELAY_THRESHOLD_MINUTES)
                          & (train[TARGET] <= 60.0)].copy()
    non_delayed_reference = float(
        train.loc[train[TARGET] <= DELAY_THRESHOLD_MINUTES, TARGET].median()
    )
    severity_preprocessor = MixedCategoricalRidgePreprocessor(
        LOW_CARDINALITY_COLUMNS, HIGH_CARDINALITY_COLUMNS, NUMERIC_COLUMNS
    )
    x_severity_train = severity_preprocessor.fit_transform(delayed_train)
    x_severity_validation = severity_preprocessor.transform(validation)
    severity_rows, severity_candidates = [], []
    delayed_validation_mask = (
        (validation[TARGET].to_numpy() > DELAY_THRESHOLD_MINUTES)
        & (validation[TARGET].to_numpy() <= 60.0)
    )
    for alpha in (0.1, 1.0, 10.0, 100.0):
        severity_model = Ridge(alpha=alpha, solver="lsqr").fit(
            x_severity_train, delayed_train[TARGET]
        )
        severity_prediction = severity_model.predict(x_severity_validation)
        hurdle_prediction = hurdle_expected_delay(
            primary_probability, severity_prediction, non_delayed_reference,
            delay_threshold_minutes=DELAY_THRESHOLD_MINUTES,
            maximum_delay_minutes=60.0,
        )
        score = compact_score(segment_metrics(
            validation[TARGET], hurdle_prediction, "two_stage_hurdle",
            "validation",
        ))
        conditional_mae = float(np.mean(np.abs(
            validation.loc[delayed_validation_mask, TARGET].to_numpy()
            - severity_prediction[delayed_validation_mask]
        )))
        severity_rows.append({"alpha": alpha,
                              "conditional_delayed_MAE": conditional_mae,
                              **score})
        severity_candidates.append((score["combined_MAE_score"],
                                    score["global_MAE"], alpha,
                                    severity_model, hurdle_prediction))
    best_severity = min(severity_candidates, key=lambda item: (item[0], item[1]))
    severity_model = best_severity[3]
    validation_predictions["two_stage_hurdle"] = best_severity[4]
    SELECTED_HYPERPARAMETERS["two_stage_hurdle"] = {
        "severity_model": "ridge_on_delays_15_to_60_min",
        "severity_alpha": float(best_severity[2]),
        "classifier": SELECTED_CLASSIFIER,
        "classifier_probability_threshold": SELECTED_CLASSIFIER_THRESHOLD,
        "non_delayed_reference_minutes": non_delayed_reference,
    }
    severity_selection = pd.DataFrame(severity_rows).sort_values(
        ["combined_MAE_score", "global_MAE"]
    )
    severity_selection.to_csv(
        REPORT_ROOT / "two_stage_severity_hyperparameter_selection.csv",
        index=False,
    )
    two_stage_validation_metrics = segment_metrics(
        validation[TARGET], validation_predictions["two_stage_hurdle"],
        "two_stage_hurdle", "validation",
    )
    two_stage_validation_metrics.to_csv(
        REPORT_ROOT / "two_stage_validation_segment_metrics.csv", index=False
    )
    two_stage_validation_haul_direction_metrics = (
        haul_direction_segment_metrics(
            validation, validation[TARGET],
            validation_predictions["two_stage_hurdle"],
            "two_stage_hurdle", "validation",
        )
    )
    validation_haul_direction_metrics = pd.concat(
        [validation_haul_direction_metrics,
         two_stage_validation_haul_direction_metrics],
        ignore_index=True,
    )
    validation_haul_direction_metrics.to_csv(
        REPORT_ROOT / "validation_haul_direction_metrics.csv", index=False
    )
    validation_metrics = pd.concat(
        [validation_metrics, two_stage_validation_metrics], ignore_index=True
    )
    validation_summary = pd.DataFrame([
        {"candidate": name, **compact_score(group)}
        for name, group in validation_metrics.groupby("model")
    ]).sort_values(["combined_MAE_score", "global_MAE"])
    validation_metrics.to_csv(
        REPORT_ROOT / "validation_segment_metrics.csv", index=False
    )
    validation_summary.to_csv(
        REPORT_ROOT / "validation_model_ranking.csv", index=False
    )
    display(severity_selection)
    display(validation_summary)
    del x_severity_train, x_severity_validation
    gc.collect()

def predict_selected_delay_probability(frame):
    selected = SELECTED_CLASSIFIERS_BY_TARGET[DELAY_THRESHOLD_MINUTES]
    name = selected["model"]
    if name.startswith("majority_baseline"):
        return np.full(len(frame), TRAIN_DELAY_RATES_BY_TARGET[DELAY_THRESHOLD_MINUTES])
    if name == "catboost_classifier":
        prepared = frame.copy()
        prepared[CATEGORICAL_COLUMNS] = (
            prepared[CATEGORICAL_COLUMNS].fillna("__MISSING__").astype(str)
        )
        prepared[NUMERIC_COLUMNS] = prepared[NUMERIC_COLUMNS].fillna(cat_medians)
        return catboost_classifier.predict_proba(
            prepared[CATEGORICAL_COLUMNS + NUMERIC_COLUMNS]
        )[:, 1]
    matrix = classifier_preprocessor.transform(frame)
    model = CLASSIFICATION_MODELS_BY_TARGET[DELAY_THRESHOLD_MINUTES][name]
    return model.predict_proba(matrix)[:, 1]


c:\Users\celti\OneDrive - Universidade de Santiago de Compostela\Verano\ML_flights_project\.venv313\Lib\site-packages\sklearn\linear_model\_sag.py:348: ConvergenceWarning: The max_iter was reached which means the coef_ did not converge
  warnings.warn(
c:\Users\celti\OneDrive - Universidade de Santiago de Compostela\Verano\ML_flights_project\.venv313\Lib\site-packages\sklearn\linear_model\_sag.py:348: ConvergenceWarning: The max_iter was reached which means the coef_ did not converge
  warnings.warn(
c:\Users\celti\OneDrive - Universidade de Santiago de Compostela\Verano\ML_flights_project\.venv313\Lib\site-packages\sklearn\linear_model\_sag.py:348: ConvergenceWarning: The max_iter was reached which means the coef_ did not converge
  warnings.warn(


,model,evaluation_scope,rows,delay_threshold_minutes,decision_threshold,actual_delay_rate,predicted_delay_rate,accuracy,balanced_accuracy,precision,recall,specificity,f1,roc_auc,average_precision,true_negative,false_positive,false_negative,true_positive
3,logistic_c1,validation,29438,15.0,0.5,0.190604,0.077111,0.829778,0.610352,0.632159,0.255748,0.964956,0.364167,0.792764,0.507948,22992,835,4176,1435
2,logistic_c0.2,validation,29438,15.0,0.5,0.190604,0.077315,0.829710,0.610446,0.631371,0.256104,0.964788,0.364397,0.792851,0.507787,22988,839,4174,1437
1,logistic_c0.05,validation,29438,15.0,0.5,0.190604,0.076364,0.829166,0.608611,0.629448,0.252183,0.965040,0.360097,0.792976,0.506930,22994,833,4196,1415
5,ridge_minutes_threshold,validation,29438,15.0,15.0,0.190604,0.108635,0.826653,0.636898,0.579425,0.330244,0.943551,0.420706,0.792338,0.503005,22482,1345,3758,1853
4,catboost_classifier,validation,29438,15.0,0.5,0.190604,0.072457,0.764046,0.499168,0.187060,0.071110,0.927225,0.103048,0.500899,0.191809,22093,1734,5212,399
0,majority_baseline,validation,29438,15.0,0.5,0.190604,0.000000,0.809396,0.500000,0.000000,0.000000,1.000000,0.000000,0.500000,0.190604,23827,0,5611,0


{'selected_classifier': 'logistic_c1', 'train_delay_rate': 0.15983749755713347}


## 8. Locked March and June evaluation

Run only after validation freezes the winner. March and June are reported
separately to expose temporal drift. A disappointing test must not be folded
back into training and retuned. Alongside the frozen Ridge contract, this section
reports the validation-weighted Ridge–CatBoost ensemble and the two-stage hurdle
model when their required components are available.

In [9]:
def predict_frozen(name, frame):
    if name == "historical_baseline":
        return baseline.predict(frame)
    if name == "ridge_catboost_ensemble":
        return weighted_prediction_blend(
            predict_frozen("ridge_without_registration", frame),
            predict_frozen("catboost", frame),
            ENSEMBLE_RIDGE_WEIGHT,
        )
    if name == "two_stage_hurdle":
        probability = predict_selected_delay_probability(frame)
        severity = severity_model.predict(severity_preprocessor.transform(frame))
        return hurdle_expected_delay(
            probability, severity, non_delayed_reference,
            delay_threshold_minutes=DELAY_THRESHOLD_MINUTES,
            maximum_delay_minutes=60.0,
        )
    if name in ridge_models:
        return ridge_models[name].predict(ridge_preprocessors[name].transform(frame))
    if name in tree_models:
        matrix, _ = dense_tree_frame(frame, tree_medians)
        return tree_models[name].predict(matrix)
    if name == "catboost":
        prepared = frame.copy()
        prepared[CATEGORICAL_COLUMNS] = (
            prepared[CATEGORICAL_COLUMNS].fillna("__MISSING__").astype(str)
        )
        prepared[NUMERIC_COLUMNS] = prepared[NUMERIC_COLUMNS].fillna(cat_medians)
        return catboost_model.predict(
            prepared[CATEGORICAL_COLUMNS + NUMERIC_COLUMNS]
        )
    raise KeyError(name)

if RUN_MODELS:
    locked_splits = {
        name: add_schedule_features(pd.read_parquet(DATA_ROOT / name))
        for name in ("test", "future_test")
    }
    assert all(not (forbidden & set(frame.columns)) for frame in locked_splits.values())
    locked_regression_models = [SELECTED_MODEL]
    if "ridge_catboost_ensemble" in validation_predictions:
        locked_regression_models.append("ridge_catboost_ensemble")
    if "two_stage_hurdle" in validation_predictions:
        locked_regression_models.append("two_stage_hurdle")
    final_metrics = pd.concat([
        segment_metrics(
            locked_splits[split_name][TARGET],
            predict_frozen(model_name, locked_splits[split_name]),
            model_name, split_name,
        )
        for model_name in locked_regression_models
        for split_name in ("test", "future_test")
    ], ignore_index=True)
    final_metrics.to_csv(REPORT_ROOT / "locked_test_metrics.csv", index=False)
    locked_haul_direction_metrics = pd.concat([
        haul_direction_segment_metrics(
            locked_splits[split_name], locked_splits[split_name][TARGET],
            predict_frozen(model_name, locked_splits[split_name]),
            model_name, split_name,
        )
        for model_name in locked_regression_models
        for split_name in ("test", "future_test")
    ], ignore_index=True)
    locked_haul_direction_metrics.to_csv(
        REPORT_ROOT / "locked_test_haul_direction_metrics.csv", index=False
    )
    display(final_metrics)

    if RUN_QUANTILE_MODEL and quantile_model is not None:
        quantile_locked_rows = []
        for split_name, frame in locked_splits.items():
            quantile_prediction = predict_dense_in_batches(
                quantile_model, frame, tree_medians
            )
            quantile_locked_rows.append(quantile_risk_metrics(
                frame[TARGET], quantile_prediction, split_name
            ))
        quantile_locked_metrics = pd.concat(quantile_locked_rows, ignore_index=True)
        quantile_locked_metrics.to_csv(
            REPORT_ROOT / "quantile_locked_test_metrics.csv", index=False
        )
        display(quantile_locked_metrics)

,model,training_scope,segment,rows,MAE,RMSE,median_absolute_error,p90_absolute_error
0,ridge,test,all,622698,9.851083,15.217367,7.189800,20.152267
1,ridge,test,punctual_<=15,501129,7.620922,10.019684,6.141227,15.797252
2,ridge,test,moderate_15_60,114386,15.893833,19.270907,13.965068,31.148719
3,ridge,test,severe_>60,7183,69.212373,84.597984,61.930314,102.790646
4,ridge,test,delayed_>15,121569,19.044201,27.790135,14.730133,37.735682
5,ridge,future_test,all,787551,10.397975,15.721337,7.659365,21.257803
6,ridge,future_test,punctual_<=15,603576,7.953702,10.455470,6.411830,16.479197
7,ridge,future_test,moderate_15_60,173092,15.377917,18.706643,13.372485,30.361096
8,ridge,future_test,severe_>60,10883,66.753429,79.103486,59.766983,101.989474
9,ridge,future_test,delayed_>15,183975,18.417024,26.445997,14.129679,36.828958


## 9. Save minute-regression contract

The bundle records the T-60 horizon, features, temporal periods and sample
fractions so later predictions cannot silently use a different contract.

In [10]:
if RUN_MODELS:
    bundle = {
        "selected_model_name": SELECTED_MODEL,
        "prediction_horizon_minutes": 60,
        "target": TARGET,
        "categorical_columns": CATEGORICAL_COLUMNS,
        "numeric_columns": NUMERIC_COLUMNS,
        "selected_hyperparameters": SELECTED_HYPERPARAMETERS.get(SELECTED_MODEL, {}),
        "train_sample_percent": TRAIN_SAMPLE_PERCENT,
        "tree_sample_percent": TREE_SAMPLE_PERCENT,
        "validation_period": "2022-12",
        "test_period": "2023-03",
        "future_test_period": "2023-06",
    }
    if SELECTED_MODEL in ridge_models:
        bundle.update({
            "model": ridge_models[SELECTED_MODEL],
            "preprocessor": ridge_preprocessors[SELECTED_MODEL],
            "categorical_columns": (
                LOW_CARDINALITY_COLUMNS + ridge_variant_columns[SELECTED_MODEL]
            ),
        })
    elif SELECTED_MODEL == "historical_baseline":
        bundle["model"] = baseline
    elif SELECTED_MODEL == "catboost":
        bundle.update({"model": catboost_model, "numeric_medians": cat_medians})
    else:
        bundle.update({"model": tree_models[SELECTED_MODEL],
                       "numeric_medians": tree_medians})
    path = MODEL_ROOT / "arrival_pre_t60_expanded_selected.joblib"
    joblib.dump(bundle, path)
    quantile_path = None
    if RUN_QUANTILE_MODEL and quantile_model is not None:
        quantile_path = MODEL_ROOT / "arrival_pre_t60_quantile_p90.joblib"
        joblib.dump({
            "model": quantile_model, "quantile": QUANTILE_LEVEL,
            "numeric_medians": tree_medians,
            "categorical_columns": CATEGORICAL_COLUMNS,
            "numeric_columns": NUMERIC_COLUMNS,
            "prediction_horizon_minutes": 60,
        }, quantile_path)
    saved_regression_extensions = {}
    if "ridge_catboost_ensemble" in validation_predictions:
        ensemble_path = MODEL_ROOT / "arrival_pre_t60_ridge_catboost_ensemble.joblib"
        joblib.dump({
            "model_type": "weighted_regression_ensemble",
            "ridge_model": ridge_models["ridge_without_registration"],
            "ridge_preprocessor": ridge_preprocessors["ridge_without_registration"],
            "catboost_model": catboost_model,
            "catboost_numeric_medians": cat_medians,
            "ridge_weight": ENSEMBLE_RIDGE_WEIGHT,
            "catboost_weight": 1.0 - ENSEMBLE_RIDGE_WEIGHT,
            "categorical_columns": CATEGORICAL_COLUMNS,
            "numeric_columns": NUMERIC_COLUMNS,
            "prediction_horizon_minutes": 60,
        }, ensemble_path)
        saved_regression_extensions["ensemble"] = str(ensemble_path)
    if "two_stage_hurdle" in validation_predictions:
        selected_classifier_name = SELECTED_CLASSIFIERS_BY_TARGET[
            DELAY_THRESHOLD_MINUTES
        ]["model"]
        if selected_classifier_name.startswith("majority_baseline"):
            classifier_component = {
                "type": "constant_probability",
                "probability": TRAIN_DELAY_RATES_BY_TARGET[DELAY_THRESHOLD_MINUTES],
            }
        elif selected_classifier_name == "catboost_classifier":
            classifier_component = {
                "type": "catboost", "model": catboost_classifier,
                "numeric_medians": cat_medians,
            }
        else:
            classifier_component = {
                "type": "logistic",
                "model": CLASSIFICATION_MODELS_BY_TARGET[
                    DELAY_THRESHOLD_MINUTES
                ][selected_classifier_name],
                "preprocessor": classifier_preprocessor,
            }
        hurdle_path = MODEL_ROOT / "arrival_pre_t60_two_stage_hurdle.joblib"
        joblib.dump({
            "model_type": "probability_times_conditional_severity",
            "classifier_name": selected_classifier_name,
            "classifier": classifier_component,
            "severity_model": severity_model,
            "severity_preprocessor": severity_preprocessor,
            "delay_threshold_minutes": DELAY_THRESHOLD_MINUTES,
            "severity_training_max_minutes": 60.0,
            "severity_prediction_cap_minutes": 60.0,
            "non_delayed_reference_minutes": non_delayed_reference,
            "prediction_horizon_minutes": 60,
        }, hurdle_path)
        saved_regression_extensions["two_stage_hurdle"] = str(hurdle_path)
    print({"saved": str(path),
           "saved_quantile": str(quantile_path) if quantile_path else None,
           "saved_extensions": saved_regression_extensions})

{'saved': 'C:\\Users\\celti\\OneDrive - Universidade de Santiago de Compostela\\Verano\\ML_flights_project\\models\\expanded\\arrival_pre_t60_expanded_selected.joblib'}


## 10. Locked classification tests and joint predictions

The classifier selected on December is frozen before March and June are opened.
Each output row contains both predictions: expected arrival-delay minutes and the
probability/binary decisions for both >15-minute and >1-minute definitions.


In [11]:
def predict_classifier_frozen_legacy(name, frame):
    if name == "majority_baseline":
        return np.full(len(frame), train_delay_rate)
    if name in logistic_candidates:
        matrix = classifier_preprocessor.transform(frame)
        return logistic_candidates[name].predict_proba(matrix)[:, 1]
    if name == "catboost_classifier":
        prepared = frame.copy()
        prepared[CATEGORICAL_COLUMNS] = (
            prepared[CATEGORICAL_COLUMNS].fillna("__MISSING__").astype(str)
        )
        prepared[NUMERIC_COLUMNS] = prepared[NUMERIC_COLUMNS].fillna(cat_medians)
        return catboost_classifier.predict_proba(
            prepared[CATEGORICAL_COLUMNS + NUMERIC_COLUMNS]
        )[:, 1]
    raise KeyError(name)

if False and RUN_MODELS and RUN_CLASSIFICATION:
    locked_classification_rows = []
    locked_classification_segment_rows = []
    for split_name, frame in locked_splits.items():
        minute_prediction = predict_frozen(SELECTED_MODEL, frame)
        delay_probability = predict_classifier_frozen_legacy(SELECTED_CLASSIFIER, frame)
        locked_classification_rows.append(delay_classification_metrics(
            frame[TARGET], delay_probability, SELECTED_CLASSIFIER, split_name,
            delay_threshold_minutes=DELAY_THRESHOLD_MINUTES,
        ))
        locked_classification_segment_rows.append(
            haul_direction_classification_metrics(
                frame, frame[TARGET], delay_probability,
                SELECTED_CLASSIFIER, split_name,
                delay_threshold_minutes=DELAY_THRESHOLD_MINUTES,
            )
        )
        locked_classification_rows.append(delay_classification_metrics(
            frame[TARGET], minute_prediction,
            f"{SELECTED_MODEL}_minutes_threshold", split_name,
            delay_threshold_minutes=DELAY_THRESHOLD_MINUTES,
            scores_are_probabilities=False,
        ))
        joint_predictions = pd.DataFrame({
            "ECTRL_ID": frame["ECTRL ID"].to_numpy(),
            "filed_off_block_time": frame["FILED OFF BLOCK TIME"].to_numpy(),
            "actual_arrival_delay_min": frame[TARGET].to_numpy(),
            "predicted_arrival_delay_min": minute_prediction,
            "actual_delayed_over_threshold": (
                frame[TARGET].to_numpy() > DELAY_THRESHOLD_MINUTES
            ),
            "predicted_delay_probability": delay_probability,
            "predicted_delayed_over_threshold": delay_probability >= 0.5,
            "duration_band": frame["Duration_Band"].to_numpy(),
            "transatlantic_direction": frame["Transatlantic_Direction"].to_numpy(),
        })
        if "ridge_catboost_ensemble" in validation_predictions:
            joint_predictions["predicted_arrival_delay_min_ensemble"] = (
                predict_frozen("ridge_catboost_ensemble", frame)
            )
        if "two_stage_hurdle" in validation_predictions:
            joint_predictions["predicted_arrival_delay_min_two_stage"] = (
                predict_frozen("two_stage_hurdle", frame)
            )
        joint_predictions.to_parquet(
            REPORT_ROOT / f"{split_name}_joint_predictions.parquet", index=False
        )
    locked_classification_metrics = pd.concat(
        locked_classification_rows, ignore_index=True
    )
    locked_classification_metrics.to_csv(
        REPORT_ROOT / "classification_locked_test_metrics.csv", index=False
    )
    pd.concat(locked_classification_segment_rows, ignore_index=True).to_csv(
        REPORT_ROOT / "classification_locked_test_haul_direction_metrics.csv",
        index=False,
    )
    display(locked_classification_metrics)

def predict_classifier_frozen(target_threshold, name, frame):
    if name.startswith("majority_baseline"):
        return np.full(len(frame), TRAIN_DELAY_RATES_BY_TARGET[target_threshold])
    if name == "catboost_classifier":
        prepared = frame.copy()
        prepared[CATEGORICAL_COLUMNS] = (
            prepared[CATEGORICAL_COLUMNS].fillna("__MISSING__").astype(str)
        )
        prepared[NUMERIC_COLUMNS] = prepared[NUMERIC_COLUMNS].fillna(cat_medians)
        return catboost_classifier.predict_proba(
            prepared[CATEGORICAL_COLUMNS + NUMERIC_COLUMNS]
        )[:, 1]
    model = CLASSIFICATION_MODELS_BY_TARGET[target_threshold][name]
    matrix = classifier_preprocessor.transform(frame)
    return model.predict_proba(matrix)[:, 1]

if RUN_MODELS and RUN_CLASSIFICATION:
    locked_classification_rows = []
    locked_classification_segment_rows = []
    for split_name, frame in locked_splits.items():
        minute_prediction = predict_frozen(SELECTED_MODEL, frame)
        selected_probabilities = {}
        for target_threshold in CLASSIFICATION_TARGETS:
            selected = SELECTED_CLASSIFIERS_BY_TARGET[target_threshold]
            name = selected["model"]
            probability_threshold = selected["probability_threshold"]
            probability = predict_classifier_frozen(target_threshold, name, frame)
            selected_probabilities[target_threshold] = probability
            locked_classification_rows.append(delay_classification_metrics(
                frame[TARGET], probability, name, split_name,
                delay_threshold_minutes=target_threshold,
                decision_threshold=probability_threshold,
            ))
            locked_classification_segment_rows.append(
                haul_direction_classification_metrics(
                    frame, frame[TARGET], probability, name, split_name,
                    delay_threshold_minutes=target_threshold,
                    decision_threshold=probability_threshold,
                )
            )
            locked_classification_rows.append(delay_classification_metrics(
                frame[TARGET], minute_prediction,
                f"{SELECTED_MODEL}_minutes_gt{target_threshold:g}", split_name,
                delay_threshold_minutes=target_threshold,
                scores_are_probabilities=False,
            ))

        primary_probability = selected_probabilities[DELAY_THRESHOLD_MINUTES]
        alternative_probability = selected_probabilities[ALTERNATIVE_DELAY_THRESHOLD_MINUTES]
        joint_predictions = pd.DataFrame({
            "ECTRL_ID": frame["ECTRL ID"].to_numpy(),
            "filed_off_block_time": frame["FILED OFF BLOCK TIME"].to_numpy(),
            "actual_arrival_delay_min": frame[TARGET].to_numpy(),
            "predicted_arrival_delay_min": minute_prediction,
            "actual_delayed_over_15_min": frame[TARGET].to_numpy() > DELAY_THRESHOLD_MINUTES,
            "predicted_delay_probability_over_15_min": primary_probability,
            "predicted_delayed_over_15_min": primary_probability >= SELECTED_CLASSIFIERS_BY_TARGET[DELAY_THRESHOLD_MINUTES]["probability_threshold"],
            "actual_delayed_over_1_min": frame[TARGET].to_numpy() > ALTERNATIVE_DELAY_THRESHOLD_MINUTES,
            "predicted_delay_probability_over_1_min": alternative_probability,
            "predicted_delayed_over_1_min": alternative_probability >= SELECTED_CLASSIFIERS_BY_TARGET[ALTERNATIVE_DELAY_THRESHOLD_MINUTES]["probability_threshold"],
            "duration_band": frame["Duration_Band"].to_numpy(),
            "transatlantic_direction": frame["Transatlantic_Direction"].to_numpy(),
        })
        if "ridge_catboost_ensemble" in validation_predictions:
            joint_predictions["predicted_arrival_delay_min_ensemble"] = (
                predict_frozen("ridge_catboost_ensemble", frame)
            )
        if "two_stage_hurdle" in validation_predictions:
            joint_predictions["predicted_arrival_delay_min_two_stage"] = (
                predict_frozen("two_stage_hurdle", frame)
            )
        joint_predictions.to_parquet(
            REPORT_ROOT / f"{split_name}_joint_predictions.parquet", index=False
        )
    locked_classification_metrics = pd.concat(
        locked_classification_rows, ignore_index=True
    )
    locked_classification_metrics.to_csv(
        REPORT_ROOT / "classification_locked_test_metrics.csv", index=False
    )
    locked_classification_metrics.to_csv(
        REPORT_ROOT / "classification_target_comparison_locked.csv", index=False
    )
    pd.concat(locked_classification_segment_rows, ignore_index=True).to_csv(
        REPORT_ROOT / "classification_locked_test_haul_direction_metrics.csv",
        index=False,
    )
    display(locked_classification_metrics)


,model,evaluation_scope,rows,delay_threshold_minutes,decision_threshold,actual_delay_rate,predicted_delay_rate,accuracy,balanced_accuracy,precision,recall,specificity,f1,roc_auc,average_precision,true_negative,false_positive,false_negative,true_positive
0,logistic_c1,test,622698,15.0,0.5,0.195229,0.069560,0.822797,0.596150,0.629574,0.224317,0.967982,0.330778,0.788472,0.495649,485084,16045,94299,27270
1,ridge_minutes_threshold,test,622698,15.0,15.0,0.195229,0.107730,0.819530,0.627973,0.568505,0.313707,0.942238,0.404311,0.786830,0.491804,472183,28946,83432,38137
2,logistic_c1,future_test,787551,15.0,0.5,0.233604,0.098593,0.788138,0.603712,0.610262,0.257562,0.949862,0.362240,0.763296,0.509875,573314,30262,136590,47385
3,ridge_minutes_threshold,future_test,787551,15.0,15.0,0.233604,0.138016,0.782617,0.625333,0.558765,0.330126,0.920540,0.415041,0.758812,0.502666,555616,47960,123240,60735


## 11. Save classification contract

The classification artefact is separate from the minute-regression artefact, so
either task can be updated or deployed without deleting the other. Separate
classification bundles are written for the >15-minute and >1-minute targets.


In [ ]:
# Legacy single-target save block retained for audit and disabled.
if False and RUN_MODELS and RUN_CLASSIFICATION:
    classifier_bundle = {
        "selected_model_name": SELECTED_CLASSIFIER,
        "prediction_horizon_minutes": 60,
        "target_definition": f"{TARGET} > {DELAY_THRESHOLD_MINUTES:g}",
        "delay_threshold_minutes": DELAY_THRESHOLD_MINUTES,
        "probability_threshold": 0.5,
        "categorical_columns": CATEGORICAL_COLUMNS,
        "numeric_columns": NUMERIC_COLUMNS,
        "selected_hyperparameters": SELECTED_HYPERPARAMETERS.get(SELECTED_CLASSIFIER, {}),
        "train_sample_percent": TRAIN_SAMPLE_PERCENT,
        "validation_period": "2022-12",
        "test_period": "2023-03",
        "future_test_period": "2023-06",
    }
    if SELECTED_CLASSIFIER == "majority_baseline":
        classifier_bundle["train_delay_rate"] = train_delay_rate
    elif SELECTED_CLASSIFIER in logistic_candidates:
        classifier_bundle.update({
            "model": logistic_candidates[SELECTED_CLASSIFIER],
            "preprocessor": classifier_preprocessor,
        })
    else:
        classifier_bundle.update({
            "model": catboost_classifier,
            "numeric_medians": cat_medians,
        })
    classifier_path = MODEL_ROOT / "arrival_pre_t60_expanded_classifier.joblib"
    joblib.dump(classifier_bundle, classifier_path)
    print({"saved_classifier": str(classifier_path)})

if RUN_MODELS and RUN_CLASSIFICATION:
    classifier_artifact_names = {
        DELAY_THRESHOLD_MINUTES: "arrival_pre_t60_expanded_classifier.joblib",
        ALTERNATIVE_DELAY_THRESHOLD_MINUTES: "arrival_pre_t60_expanded_classifier_gt1min.joblib",
    }
    saved_classifier_paths = {}
    for target_threshold, artifact_name in classifier_artifact_names.items():
        selected = SELECTED_CLASSIFIERS_BY_TARGET[target_threshold]
        selected_name = selected["model"]
        classifier_bundle = {
            "selected_model_name": selected_name,
            "prediction_horizon_minutes": 60,
            "target_definition": f"{TARGET} > {target_threshold:g}",
            "delay_threshold_minutes": target_threshold,
            "probability_threshold": selected["probability_threshold"],
            "threshold_selection_objective": THRESHOLD_SELECTION_OBJECTIVE,
            "false_negative_cost": FALSE_NEGATIVE_COST,
            "false_positive_cost": FALSE_POSITIVE_COST,
            "categorical_columns": CATEGORICAL_COLUMNS,
            "numeric_columns": NUMERIC_COLUMNS,
            "selected_hyperparameters": SELECTED_HYPERPARAMETERS.get(selected_name, {}),
            "train_sample_percent": TRAIN_SAMPLE_PERCENT,
            "validation_period": "2022-12",
            "test_period": "2023-03",
            "future_test_period": "2023-06",
        }
        if selected_name.startswith("majority_baseline"):
            classifier_bundle["train_delay_rate"] = TRAIN_DELAY_RATES_BY_TARGET[target_threshold]
        elif selected_name == "catboost_classifier":
            classifier_bundle.update({
                "model": catboost_classifier, "numeric_medians": cat_medians,
            })
        else:
            classifier_bundle.update({
                "model": CLASSIFICATION_MODELS_BY_TARGET[target_threshold][selected_name],
                "preprocessor": classifier_preprocessor,
            })
        classifier_path = MODEL_ROOT / artifact_name
        joblib.dump(classifier_bundle, classifier_path)
        saved_classifier_paths[f"gt{target_threshold:g}min"] = str(classifier_path)
    print({"saved_classifiers": saved_classifier_paths})


{'saved_classifier': 'C:\\Users\\celti\\OneDrive - Universidade de Santiago de Compostela\\Verano\\ML_flights_project\\models\\expanded\\arrival_pre_t60_expanded_classifier.joblib'}


: 

## Guardrails

- Lower validation error does not prove future reliability; report both tests.
- The new months improve coverage but remain non-consecutive snapshots.
- Weather stays deferred until this expanded flight-only baseline is frozen.
- Primary classification means arrival delay >15 minutes; exactly 15 remains OTP15.
- The >1-minute experiment is a secondary sensitivity analysis, not a replacement for OTP15.
- Accuracy must be read with recall, precision and PR-AUC because classes are imbalanced.
- CNN/LSTM remains secondary until dense continuous sequences are available.